# TDM GHG Calculator — Mixed-Use Development Demo

This notebook demonstrates the `tdm_ghg` library on a **theoretical mixed-use urban development** at the Plan/Community scale.

## Scenario

A city is planning a mixed-use transit corridor redevelopment with the following TDM strategies:

| # | Strategy | Subsector |
|---|----------|-----------|
| T-20 | Expand Bikeway Network | Neighborhood Design |
| T-22-B | Implement Electric Bikeshare | Neighborhood Design |
| T-26 | Increase Transit Service Frequency | Transit |
| T-46 | Provide Transit Shelters (with real-time info) | Transit |

We will:
1. Compute each strategy's individual GHG reduction
2. Combine strategies within each subsector (with CAPCOA caps)
3. Combine across subsectors using the multi-subsector cap (70%)

## Setup

Add the project root to the Python path so we can import `tdm_ghg` directly from the repo.

In [ ]:
import sys, pathlib

# Ensure the repo root is on the path so tdm_ghg can be imported without installing
repo_root = str(pathlib.Path.cwd().parent)
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from tdm_ghg import (
    TDMContext, Scale, LocationType, LandUseType,
    t20_expand_bikeway_network,
    t22b_implement_electric_bikeshare,
    t26_increase_transit_service_frequency,
    t46_provide_transit_shelters,
    run_neighborhood_design,
    run_transit,
    run_multi_subsector,
    multiplicative_dampening,
    SUBSECTOR_CAPS,
    MULTI_SUBSECTOR_CAP,
    registry,
)

import pandas as pd

print("tdm_ghg loaded successfully")
print(f"Registered measures: {len(registry.measures)}")

## 1. Define the Analysis Context

Our project is a **Plan/Community** scale analysis in an **urban** setting with **mixed** land use. We load all strategy parameters into the context's `params` dict.

In [ ]:
ctx = TDMContext(
    scale=Scale.PLAN_COMMUNITY,
    location_type=LocationType.URBAN,
    land_use_type=LandUseType.MIXED,
    params={
        # --- T-20: Expand Bikeway Network ---
        # The city currently has 40 miles of bikeways and plans to add 30 more.
        "existing_bikeway_miles_in_community": 40.0,
        "proposed_bikeway_miles_in_community": 70.0,   # 75% increase
        "bike_mode_share": 0.018,                      # 1.8% existing bike share
        "vehicle_mode_share": 0.82,                    # 82% auto share
        "average_oneway_bicycle_trip_length": 2.3,     # miles
        "average_oneway_vehicle_trip_length": 9.7,     # miles (LA CBSA)

        # --- T-22-B: Electric Bikeshare ---
        # New e-bikeshare system covering 45% of residences in the plan area.
        "pct_residences_with_access_with_measure": 0.45,

        # --- T-26: Increase Transit Frequency ---
        # Double bus frequency on 60% of routes in the corridor.
        "pct_increase_in_transit_frequency": 1.0,      # 100% increase = double
        "level_of_implementation": 0.60,               # 60% of routes
        "transit_mode_share": 0.045,                   # 4.5% transit share

        # --- T-46: Transit Shelters ---
        # Install shelters with real-time arrival info at 25 stops.
        "num_stops_with_new_shelters": 25,
        "avg_boardings_per_day_at_improved_stops": 180.0,
        "avg_boardings_per_day_across_agency": 55000.0,
        "include_real_time_information": True,
    },
)

print(f"Scale:         {ctx.scale.value}")
print(f"Location:      {ctx.location_type.value}")
print(f"Land use:      {ctx.land_use_type.value}")
print(f"Parameters:    {len(ctx.params)} values loaded")

## 2. Individual Strategy Results

Call each measure function directly to see its standalone GHG reduction.

In [ ]:
# T-20: Expand Bikeway Network
t20_result = t20_expand_bikeway_network(
    existing_bikeway_miles_in_community=40.0,
    proposed_bikeway_miles_in_community=70.0,
    bike_mode_share=0.018,
    vehicle_mode_share=0.82,
    average_oneway_bicycle_trip_length=2.3,
    average_oneway_vehicle_trip_length=9.7,
)

# T-22-B: Electric Bikeshare
t22b_result = t22b_implement_electric_bikeshare(
    pct_residences_with_access_with_measure=0.45,
)

# T-26: Increase Transit Frequency
t26_result = t26_increase_transit_service_frequency(
    pct_increase_in_transit_frequency=1.0,
    level_of_implementation=0.60,
    transit_mode_share=0.045,
    vehicle_mode_share=0.82,
)

# T-46: Transit Shelters with real-time info
t46_result = t46_provide_transit_shelters(
    num_stops_with_new_shelters=25,
    avg_boardings_per_day_at_improved_stops=180.0,
    avg_boardings_per_day_across_agency=55000.0,
    transit_mode_share=0.045,
    include_real_time_information=True,
)

# Summarize
individual = pd.DataFrame([
    {"Measure": "T-20",   "Strategy": "Expand Bikeway Network",         "Subsector": "Neighborhood Design", "Reduction": t20_result},
    {"Measure": "T-22-B", "Strategy": "Electric Bikeshare",             "Subsector": "Neighborhood Design", "Reduction": t22b_result},
    {"Measure": "T-26",   "Strategy": "Increase Transit Frequency",     "Subsector": "Transit",             "Reduction": t26_result},
    {"Measure": "T-46",   "Strategy": "Transit Shelters + RTI",         "Subsector": "Transit",             "Reduction": t46_result},
])
individual["Reduction (%)"] = individual["Reduction"].map(lambda x: f"{x:.4%}")
individual

## 3. Subsector-Level Combination

CAPCOA requires combining measures within each subsector using **multiplicative dampening** before applying the subsector cap. The `run_*` orchestrators handle this automatically using the context.

In [ ]:
# Neighborhood Design subsector (cap 10%)
nd_cap = SUBSECTOR_CAPS[("plan_community", "neighborhood_design")]
nd_result = run_neighborhood_design(ctx)

print("=== Neighborhood Design Subsector ===")
print(f"  T-20 individual:   {t20_result:.4%}")
print(f"  T-22-B individual: {t22b_result:.4%}")
print(f"  Subsector cap:     {nd_cap:.0%}")
print(f"  Combined result:   {nd_result:.4%}")
print()

# Transit subsector (cap 15%) — using standard measures (not BRT)
tr_cap = SUBSECTOR_CAPS[("plan_community", "transit")]
tr_result = run_transit(ctx, use_brt=False)

print("=== Transit Subsector ===")
print(f"  T-26 individual:   {t26_result:.4%}")
print(f"  T-46 individual:   {t46_result:.4%}")
print(f"  Subsector cap:     {tr_cap:.0%}")
print(f"  Combined result:   {tr_result:.4%}")

## 4. Multi-Subsector GHG Combination

Per CAPCOA, Land Use + Neighborhood Design + Parking Management + Transit are combined with multiplicative dampening and a **70% overall cap**. Trip Reduction and School Programs are excluded (they address separate VMT categories).

Since our scenario only includes Neighborhood Design and Transit strategies, the other subsectors contribute 0.

In [ ]:
# Multi-subsector combination via the orchestrator
total = run_multi_subsector(ctx, use_brt=False)

print("=== Multi-Subsector Summary ===")
print(f"  Neighborhood Design: {nd_result:.4%}")
print(f"  Transit:             {tr_result:.4%}")
print(f"  Land Use:             0.00%  (no measures in this scenario)")
print(f"  Parking Management:   0.00%  (not yet implemented)")
print(f"  Multi-subsector cap: {MULTI_SUBSECTOR_CAP:.0%}")
print(f"  ─────────────────────────────")
print(f"  TOTAL GHG REDUCTION: {total:.4%}")

## 5. Understanding Multiplicative Dampening

Simple addition would overcount overlapping effectiveness. Multiplicative dampening uses:

$$\text{Combined} = \min\!\bigl(\text{cap},\; 1 - \prod_i (1 - r_i)\bigr)$$

Let's compare naive addition vs. dampening for our transit strategies.

In [ ]:
transit_reductions = [t26_result, t46_result]

naive_sum = sum(transit_reductions)
dampened = multiplicative_dampening(transit_reductions, max_reduction_percentage=-tr_cap)

comparison = pd.DataFrame({
    "Method": ["Naive addition", "Multiplicative dampening (capped)"],
    "Combined reduction": [naive_sum, dampened],
    "Formatted": [f"{naive_sum:.4%}", f"{dampened:.4%}"],
})
comparison

## 6. Sensitivity: What If We Use BRT Instead?

T-28 (Bus Rapid Transit) is **mutually exclusive** with T-26, T-27, and T-46. The `run_transit` orchestrator handles this via the `use_brt` flag. Let's compare.

In [ ]:
# BRT scenario — same context params work because T-28 shares
# pct_increase_in_transit_frequency and level_of_implementation with T-26
from tdm_ghg import t28_provide_bus_rapid_transit

t28_result = t28_provide_bus_rapid_transit(
    pct_increase_in_transit_frequency=1.0,
    level_of_implementation=0.60,
    transit_mode_share=0.045,
    vehicle_mode_share=0.82,
)

tr_brt = run_transit(ctx, use_brt=True)
total_brt = run_multi_subsector(ctx, use_brt=True)

scenarios = pd.DataFrame([
    {"Scenario": "Standard (T-26 + T-46)", "Transit reduction": f"{tr_result:.4%}", "Total GHG reduction": f"{total:.4%}"},
    {"Scenario": "BRT (T-28)",             "Transit reduction": f"{tr_brt:.4%}",    "Total GHG reduction": f"{total_brt:.4%}"},
])
scenarios

## 7. Registry Inspection

The registry tracks metadata for every measure. Let's see which measures are applicable to our context.

In [ ]:
applicable = registry.filter(ctx)

reg_df = pd.DataFrame([
    {
        "ID": m.measure_id,
        "Name": m.name,
        "Subsector": m.subsector,
        "Max": f"{m.measure_max:.1%}",
        "Excludes": ", ".join(sorted(m.mutually_exclusive_with)) or "—",
    }
    for m in applicable
])

print(f"{len(applicable)} measures applicable to urban mixed-use (plan/community):\n")
reg_df

---

## Summary

This notebook demonstrated:

- **Individual measure evaluation** — calling each strategy function with project-specific inputs
- **Subsector combination** — `run_neighborhood_design()` and `run_transit()` apply multiplicative dampening with CAPCOA caps (10% and 15% respectively)
- **Multi-subsector combination** — `run_multi_subsector()` combines all subsectors under a 70% cap
- **Sensitivity analysis** — comparing standard transit improvements against BRT
- **Registry inspection** — discovering which measures apply to a given context

All formulas and default values trace back to the CAPCOA 2024 Handbook, Transportation Section.